# CatBoost fold0 — ctr 정규화 묶음 (exp_068)

Driver native + max_ctr_complexity=1 + Counter prior 다양화(1→3) + model_size_reg=1.0 + store_all_simple_ctr=True + ctr_leaf_count_limit=100000. task_type=GPU(exp_025 동일환경). 게이트: vs exp_025 fold0 0.951265.

In [ ]:
# 1) input 자동탐색 (마운트 비표준: /kaggle/input/{datasets,competitions}/...) — torch import 前
import sys, os, glob, subprocess
from pathlib import Path
print('/kaggle/input:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'NONE')
c = glob.glob('/kaggle/input/**/src/config.py', recursive=True); assert c, 'src/config.py 못 찾음'
SRC_ROOT = str(Path(c[0]).parents[1]); print('SRC_ROOT:', SRC_ROOT)
cc = glob.glob('/kaggle/input/**/playground-series-s6e5', recursive=True); assert cc, '대회 폴더 못 찾음'
COMP = Path(cc[0]); print('COMP:', COMP)
ac = glob.glob('/kaggle/input/**/f1_strategy_dataset*.csv', recursive=True); assert ac, '증강 csv 못 찾음'
AUG = Path(ac[0]); print('AUG:', AUG)

In [ ]:
# 2) deps(hydra) + CatBoost/GPU 확인
import subprocess, sys
def pip(*a): subprocess.run([sys.executable,'-m','pip','install','-q',*a], check=True)
pip('hydra-core','python-dotenv')
print('GPU:', subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True).stdout.strip())
import catboost; print('catboost', catboost.__version__)

In [ ]:
# 3) src import
import sys
sys.path.insert(0, SRC_ROOT)
from src import config
from src.train_catboost import run
print('import OK:', config.__file__)

In [ ]:
# 4) 경로 override
config.TRAIN_PATH = COMP / 'train.csv'
config.TEST_PATH = COMP / 'test.csv'
config.SAMPLE_SUBMISSION_PATH = COMP / 'sample_submission.csv'
config.SOURCE_AUG_PATH = AUG
out = Path('/kaggle/working')
config.OOF_DIR = out / 'oof'; config.SUBMISSION_DIR = out / 'submissions'; config.LOG_DIR = out / 'logs'
import pandas as pd
_a = pd.read_csv(config.SOURCE_AUG_PATH); print('AUG shape:', _a.shape)
assert len(_a) == 101371, f'증강 행수 불일치: {len(_a)}'
assert config.TRAIN_PATH.exists(), f'train.csv 없음: {config.TRAIN_PATH}'

In [ ]:
# 5) cfg + run — CatBoost ctr 정규화 묶음, fold0
from omegaconf import OmegaConf
import time
CONF = Path(SRC_ROOT) / 'conf'
mc = OmegaConf.load(CONF / 'model' / 'catboost.yaml')
mc.num_boost_round = 15000   # exp_025 통제(default 5000→15000, cap 미완 회피)
mc.params['max_ctr_complexity'] = 1
mc.params['simple_ctr'] = [
  'Borders:CtrBorderCount=15:CtrBorderType=Uniform:TargetBorderCount=1:TargetBorderType=MinEntropy:Prior=0/1:Prior=0.5/1:Prior=1/1',
  'Counter:CtrBorderCount=15:CtrBorderType=Uniform:Prior=0/1:Prior=0.5/1:Prior=1/1',
]
mc.params['model_size_reg'] = 1.0
mc.params['store_all_simple_ctr'] = True
mc.params['ctr_leaf_count_limit'] = 100000
cfg = OmegaConf.create({
    'exp_id': 'exp_068_cat_ctrreg_fold0',
    'notes': 'CatBoost fold0: ctr 정규화 묶음(Counter prior+model_size_reg+store_all+leaf_limit) vs exp_025 0.951265',
    'use_wandb': False, 'max_folds': 1,
    'model': mc,
    'features': OmegaConf.load(CONF / 'features' / 'base_yearcat.yaml'),
    'augment': {'enabled': True, 'weight': 1.0},
})
print(OmegaConf.to_yaml(cfg))
t0=time.time(); result=run(cfg); print(result, f'{time.time()-t0:.0f}s')
print('fold0 AUC =', result.get('fold_scores',[None])[0], '| exp_025 fold0 0.951265')